In [2]:
!pip install yfinance pandas

import yfinance as yf
import pandas as pd


[notice] A new release of pip is available: 23.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import yfinance as yf

def analyze_stock(ticker, period="1y"):

    df = yf.Ticker(ticker).history(period=period)

    if df.empty:
        print(f"{ticker}: No data found.\n")
        return

    # ------------------------------------------------------
    # ICHIMOKU LINES
    # ------------------------------------------------------
    df['conversion'] = (df['High'].rolling(9).max() + df['Low'].rolling(9).min()) / 2     # Tenkan
    df['base'] = (df['High'].rolling(26).max() + df['Low'].rolling(26).min()) / 2         # Kijun
    df['spanA'] = ((df['conversion'] + df['base']) / 2).shift(26)                        # Span A
    df['spanB'] = ((df['High'].rolling(52).max() + df['Low'].rolling(52).min()) / 2).shift(26) # Span B
    df['chikou'] = df['Close'].shift(-26)

    # Current values
    c = df['Close'].iloc[-1]
    conv = df['conversion'].iloc[-1]
    base = df['base'].iloc[-1]
    spanA = df['spanA'].iloc[-1]
    spanB = df['spanB'].iloc[-1]

    cloud_color = "BULLISH (Green)" if spanA > spanB else "BEARISH (Red)"

    # ------------------------------------------------------
    # PRICE LOCATION VS CLOUD
    # ------------------------------------------------------
    if c > spanA and c > spanB:
        price_zone = "Above Cloud (Strong Bullish)"
        market_mode = "BULL"
    elif c < spanA and c < spanB:
        price_zone = "Below Cloud (Strong Bearish)"
        market_mode = "BEAR"
    else:
        price_zone = "Inside Cloud (No Direction)"
        market_mode = "NEUTRAL"

    # ------------------------------------------------------
    # TK CROSS
    # ------------------------------------------------------
    if conv > base:
        tk = "Bullish TK Cross — rising momentum"
    elif conv < base:
        tk = "Bearish TK Cross — weakening momentum"
    else:
        tk = "Neutral TK Cross"

    # ------------------------------------------------------
    # ICHIMOKU SIGNAL
    # ------------------------------------------------------
    if (market_mode == "BULL" and conv > base and spanA > spanB):
        ichi_signal = "STRONG BUY"
    elif (market_mode == "BEAR" and conv < base and spanA < spanB):
        ichi_signal = "STRONG SELL"
    else:
        ichi_signal = "NEUTRAL / WAIT"

    # ------------------------------------------------------
    # RSI
    # ------------------------------------------------------
    delta = df['Close'].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain.iloc[-1] / avg_loss.iloc[-1] if avg_loss.iloc[-1] != 0 else 999
    rsi = 100 - (100 / (1 + rs))

    if rsi > 70:
        rsi_status = "Overbought — avoid buying"
    elif rsi < 30:
        rsi_status = "Oversold — dip buying zone"
    else:
        rsi_status = "Normal"

    # ------------------------------------------------------
    # MACD
    # ------------------------------------------------------
    ema12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema26 = df['Close'].ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    signal = macd.ewm(span=9, adjust=False).mean()
    macd_hist = macd - signal
    macd_desc = "Positive → bullish momentum" if macd_hist.iloc[-1] > 0 else "Negative → bearish momentum"

    # ------------------------------------------------------
    # ATR (14)
    # ------------------------------------------------------
    df['TR'] = df['High'] - df['Low']
    df['ATR'] = df['TR'].rolling(14).mean()
    atr = df['ATR'].iloc[-1]

    # safe formatter
    def fmt(x):
        return f"{x:.2f}" if x is not None else "—"

    # ------------------------------------------------------
    # BUY / SELL LOGIC (final corrected)
    # ------------------------------------------------------
    if market_mode == "BULL":
        aggressive_buy = c if rsi < 70 else None
        conservative_buy = min(base, c)
        stop_loss = conservative_buy - 1.5 * atr

        suggestion = (
            f"Aggressive Buy: {fmt(aggressive_buy)}\n"
            f"Conservative Buy (pullback): {fmt(conservative_buy)}\n"
            f"Stop-Loss: {fmt(stop_loss)}"
        )

    elif market_mode == "BEAR":
        aggressive_sell = c if rsi > 30 else None
        conservative_sell = max(base, c)
        stop_loss = conservative_sell + 1.5 * atr

        suggestion = (
            f"Aggressive Sell: {fmt(aggressive_sell)}\n"
            f"Conservative Sell (pullback): {fmt(conservative_sell)}\n"
            f"Stop-Loss: {fmt(stop_loss)}"
        )

    else:
        suggestion = "Price inside cloud — wait."

    # ------------------------------------------------------
    # PRINT OUTPUT
    # ------------------------------------------------------
    print("\n-----------------------------------------------")
    print(f"📈 {ticker} — Analysis")
    print("-----------------------------------------------")
    print(f"Price: {c:.2f}")
    print(f"Cloud: {cloud_color}")
    print(f"Position: {price_zone}")
    print(f"TK Cross: {tk}")
    print(f"Ichimoku Signal: {ichi_signal}")
    print(f"Action:\n{suggestion}")
    print(f"RSI: {rsi:.2f} → {rsi_status}")
    print(f"MACD Hist: {macd_hist.iloc[-1]:.4f} → {macd_desc}")
    print("-----------------------------------------------\n")


# ------------------------------------------------------
# RUN MULTIPLE STOCKS
# ------------------------------------------------------
tickers = ["LGEINDIA.NS", "CUPID.NS", "ADANIPOWER.NS", "GROWW.NS"]

for t in tickers:
    analyze_stock(t)



-----------------------------------------------
📈 LGEINDIA.NS — Analysis
-----------------------------------------------
Price: 1617.80
Cloud: BEARISH (Red)
Position: Inside Cloud (No Direction)
TK Cross: Neutral TK Cross
Ichimoku Signal: NEUTRAL / WAIT
Action:
Price inside cloud — wait.
RSI: 42.06 → Normal
MACD Hist: 1.0413 → Positive → bullish momentum
-----------------------------------------------


-----------------------------------------------
📈 CUPID.NS — Analysis
-----------------------------------------------
Price: 312.84
Cloud: BULLISH (Green)
Position: Above Cloud (Strong Bullish)
TK Cross: Bullish TK Cross — rising momentum
Ichimoku Signal: STRONG BUY
Action:
Aggressive Buy: —
Conservative Buy (pullback): 262.29
Stop-Loss: 239.77
RSI: 81.03 → Overbought — avoid buying
MACD Hist: 5.1044 → Positive → bullish momentum
-----------------------------------------------


-----------------------------------------------
📈 ADANIPOWER.NS — Analysis
---------------------------------

In [3]:
import yfinance as yf
import numpy as np

def analyze_stock(ticker, period="1y"):

    df = yf.Ticker(ticker).history(period=period)

    if df.empty:
        print(f"{ticker}: No data found.\n")
        return

    # ------------------------------------------------------
    # ICHIMOKU LINES
    # ------------------------------------------------------
    df['conversion'] = (df['High'].rolling(9).max() + df['Low'].rolling(9).min()) / 2     # Tenkan
    df['base'] = (df['High'].rolling(26).max() + df['Low'].rolling(26).min()) / 2         # Kijun
    df['spanA'] = ((df['conversion'] + df['base']) / 2).shift(26)                        # Leading Span A
    df['spanB'] = ((df['High'].rolling(52).max() + df['Low'].rolling(52).min()) / 2).shift(26) # Leading Span B
    df['chikou'] = df['Close'].shift(-26)                                                # Lagging Span

    c = df['Close'].iloc[-1]       # current price
    conv = df['conversion'].iloc[-1]
    base = df['base'].iloc[-1]
    spanA = df['spanA'].iloc[-1]
    spanB = df['spanB'].iloc[-1]

    cloud_color = "BULLISH (Green)" if spanA > spanB else "BEARISH (Red)"

    # ------------------------------------------------------
    # PRICE LOCATION vs CLOUD
    # ------------------------------------------------------
    if c > spanA and c > spanB:
        price_zone = "Above Cloud (Strong Bullish)"
        market_mode = "BULL"
    elif c < spanA and c < spanB:
        price_zone = "Below Cloud (Strong Bearish)"
        market_mode = "BEAR"
    else:
        price_zone = "Inside Cloud (No Direction)"
        market_mode = "NEUTRAL"

    # ------------------------------------------------------
    # TK CROSS
    # ------------------------------------------------------
    if conv > base:
        tk = "Bullish TK Cross — rising momentum"
    elif conv < base:
        tk = "Bearish TK Cross — weakening momentum"
    else:
        tk = "Neutral TK Cross"

    # ------------------------------------------------------
    # ICHIMOKU SUMMARY SIGNAL
    # ------------------------------------------------------
    if (market_mode == "BULL" and conv > base and spanA > spanB):
        ichi_signal = "STRONG BUY"
    elif (market_mode == "BEAR" and conv < base and spanA < spanB):
        ichi_signal = "STRONG SELL"
    else:
        ichi_signal = "NEUTRAL / WAIT"

    # ------------------------------------------------------
    # ATR (volatility)
    # ------------------------------------------------------
    df['ATR'] = (df['High'] - df['Low']).rolling(14).mean()
    atr = df['ATR'].iloc[-1]

    # ------------------------------------------------------
    # RSI
    # ------------------------------------------------------
    delta = df['Close'].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain.iloc[-1] / avg_loss.iloc[-1] if avg_loss.iloc[-1] != 0 else 999
    rsi = 100 - (100 / (1 + rs))

    if np.isnan(rsi):
        rsi_status = "Normal"
    elif rsi > 70:
        rsi_status = "Overbought — avoid buying"
    elif rsi < 30:
        rsi_status = "Oversold — dip buying zone"
    else:
        rsi_status = "Normal"

    # ------------------------------------------------------
    # MACD
    # ------------------------------------------------------
    ema12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema26 = df['Close'].ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    signal = macd.ewm(span=9, adjust=False).mean()
    macd_hist = macd - signal

    macd_desc = (
        "Positive → bullish momentum"
        if macd_hist.iloc[-1] > 0 else
        "Negative → bearish momentum"
    )

    # ------------------------------------------------------
    # BUY / SELL LOGIC (includes new inside-cloud signals)
    # ------------------------------------------------------
    if market_mode == "BULL":

        # Aggressive only if NOT overbought
        aggressive_buy = c if rsi < 70 else None

        # Conservative = Kijun pullback
        conservative_buy = min(base, c)

        # Stop-loss below conservative entry
        stop_loss = conservative_buy - 1.5 * atr

        suggestion = (
            f"Aggressive Buy: {aggressive_buy:.2f}" if aggressive_buy else "Aggressive Buy: —"
        )
        suggestion += (
            f"\nConservative Buy (pullback): {conservative_buy:.2f}"
            f"\nStop-Loss: {stop_loss:.2f}"
        )

    elif market_mode == "BEAR":

        aggressive_sell = c
        conservative_sell = max(base, c)
        stop_loss = conservative_sell + 1.5 * atr

        suggestion = (
            f"Aggressive Sell: {aggressive_sell:.2f}"
            f"\nConservative Sell (pullback): {conservative_sell:.2f}"
            f"\nStop-Loss: {stop_loss:.2f}"
        )

    else:
        # 💡 NEW: Inside cloud → show breakout levels
        cloud_top = max(spanA, spanB)
        cloud_bottom = min(spanA, spanB)

        suggestion = (
            "Inside cloud — wait.\n"
            f"Potential Buy Above Cloud: {cloud_top:.2f}\n"
            f"Potential Sell Below Cloud: {cloud_bottom:.2f}"
        )

    # ------------------------------------------------------
    # PRINT RESULTS
    # ------------------------------------------------------
    print("\n-----------------------------------------------")
    print(f"📈 {ticker} — Analysis")
    print("-----------------------------------------------")
    print(f"Price: {c:.2f}")
    print(f"Cloud: {cloud_color}")
    print(f"Position: {price_zone}")
    print(f"TK Cross: {tk}")
    print(f"Ichimoku Signal: {ichi_signal}")
    print("Action:")
    print(suggestion)
    print(f"RSI: {rsi:.2f} → {rsi_status}")
    print(f"MACD Hist: {macd_hist.iloc[-1]:.4f} → {macd_desc}")
    print("-----------------------------------------------\n")


tickers = ["LGEINDIA.NS","CUPID.NS","ADANIPOWER.NS","GROWW.NS", "TSLA"]

for t in tickers:
    analyze_stock(t)



-----------------------------------------------
📈 LGEINDIA.NS — Analysis
-----------------------------------------------
Price: 1617.80
Cloud: BEARISH (Red)
Position: Inside Cloud (No Direction)
TK Cross: Neutral TK Cross
Ichimoku Signal: NEUTRAL / WAIT
Action:
Inside cloud — wait.
Potential Buy Above Cloud: nan
Potential Sell Below Cloud: nan
RSI: 42.06 → Normal
MACD Hist: 1.0413 → Positive → bullish momentum
-----------------------------------------------


-----------------------------------------------
📈 CUPID.NS — Analysis
-----------------------------------------------
Price: 312.84
Cloud: BULLISH (Green)
Position: Above Cloud (Strong Bullish)
TK Cross: Bullish TK Cross — rising momentum
Ichimoku Signal: STRONG BUY
Action:
Aggressive Buy: —
Conservative Buy (pullback): 262.29
Stop-Loss: 239.77
RSI: 81.03 → Overbought — avoid buying
MACD Hist: 5.1044 → Positive → bullish momentum
-----------------------------------------------


-----------------------------------------------
📈 A